# IE 7615 · Discriminative Project — Milestone 1
### Single-object identification across 73 classes: custom CNN vs. transfer learning

**Study Group 6:** Sankalp Hegde · Reshma Jose · Jairaghavendra Sridhar

This notebook contains the full pipeline used for every model in the report: dataset cleaning and the shared stratified split, data augmentation, all six architectures, training with early stopping, evaluation, and the final comparison.

| Run | Architecture | Mode |
|---|---|---|
| `simple_cnn_scratch` | Custom 3-block CNN | trained from scratch |
| `convnext_tiny_ft` | ConvNeXt-Tiny | fine-tuned |
| `resnet50_frozen` / `resnet50_ft` | ResNet50 | frozen / fine-tuned |
| `efficientnet_b0_frozen` / `efficientnet_b0_ft` | EfficientNet-B0 | frozen / fine-tuned |
| `mobilenet_v2_frozen` / `mobilenet_v2_ft` | MobileNetV2 | frozen / fine-tuned |
| `densenet121_ft` | DenseNet121 | fine-tuned |

**Run on Google Colab** with a T4 GPU (Runtime → Change runtime type → T4 GPU), then set `DRIVE_DATA_DIR` in the configuration cell to the folder containing `images_OBJ001 … images_OBJ073`. Each run writes `results_<tag>.json`, training curves, a confusion matrix, per-image predictions and the best weights to `DRIVE_OUT_DIR/results/`.

## Environment & GPU Check

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## Mount Google Drive & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ======================= CONFIG =======================
DRIVE_DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/IE7615 Deep Learning for AI"
DRIVE_OUT_DIR = "/content/drive/MyDrive/IE7615_Project/milestone1"

REBUILD_SPLIT = False

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
PATIENCE = 5
SEED = 42
LR = {"scratch": 1e-3, "frozen": 1e-3, "finetune": 1e-4}

import os
assert os.path.exists(DRIVE_DATA_DIR), f"Path not found: {DRIVE_DATA_DIR} — fix DRIVE_DATA_DIR above"
print("Found dataset directory.")

contents = sorted(os.listdir(DRIVE_DATA_DIR))
print(f"\nNumber of items: {len(contents)}")
print("First 5:", contents[:5])
print("Last 5:", contents[-5:])

## Imports & Setup

In [ ]:
import os, sys, json, time, random, shutil, hashlib, collections
from pathlib import Path

OUT_DIR = Path(DRIVE_OUT_DIR)
DATA_DIR = Path("/content/data_single")
if not DATA_DIR.exists():
    print("Copying dataset from Drive to local Colab disk (faster for training)...")
    shutil.copytree(DRIVE_DATA_DIR, DATA_DIR)
    print("Copy complete.")
else:
    print("Local copy already exists, skipping copy.")

RESULTS_DIR = OUT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 2

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
import matplotlib.pyplot as plt

Image.MAX_IMAGE_PIXELS = None


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"Data:   {DATA_DIR}")
print(f"Output: {OUT_DIR}")

## Train / Val / Test Split

In [ ]:
IMG_EXTS = {".jpg", ".jpeg"}


def scan_dataset(root):
    """One folder per object (images_OBJ001, ...) -> DataFrame(path, label).

    Rules:
      * label = folder name without "images_" prefix (e.g. OBJ001)
      * only files directly inside each folder
      * *_bg_* background-only photos are excluded (Milestone 2 material)
      * byte-identical duplicates kept once (prevents train/test leakage)
    """
    rows, skipped, seen = [], collections.Counter(), set()
    class_dirs = sorted(p for p in Path(root).iterdir() if p.is_dir() and not p.name.startswith("."))
    for class_dir in class_dirs:
        label = class_dir.name.removeprefix("images_")
        for f in sorted(class_dir.iterdir()):
            if f.is_dir():
                if not f.name.startswith("."):
                    skipped["nested folder (ignored)"] += sum(1 for _ in f.rglob("*.*"))
                continue
            if f.name.startswith(".") or f.suffix.lower() not in IMG_EXTS:
                if not f.name.startswith("."):
                    skipped["unsupported file type"] += 1
                continue
            if "_bg_" in f.name.lower():
                skipped["background-only (_bg_)"] += 1
                continue
            digest = hashlib.md5(f.read_bytes()).hexdigest()
            if (label, digest) in seen:
                skipped["duplicate image"] += 1
                continue
            try:
                with Image.open(f) as im:
                    im.verify()
            except Exception:
                skipped["corrupt"] += 1
                continue
            seen.add((label, digest))
            rows.append({"path": str(f.relative_to(root)), "label": label})
    for reason, n in skipped.items():
        print(f"Skipped {n:4d}  {reason}")
    return pd.DataFrame(rows)


def make_split(df, seed):
    train, rest = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=seed)
    val, test = train_test_split(rest, test_size=0.50, stratify=rest["label"], random_state=seed)
    return pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")],
                     ignore_index=True)


SPLIT_PATH = OUT_DIR / "split.csv"
all_df = scan_dataset(DATA_DIR)

if REBUILD_SPLIT or not SPLIT_PATH.exists():
    split_df = make_split(all_df, SEED)
    split_df.to_csv(SPLIT_PATH, index=False)
    print(f"Created {SPLIT_PATH}")
else:
    split_df = pd.read_csv(SPLIT_PATH)
    missing = set(split_df["path"]) - set(all_df["path"])
    new = set(all_df["path"]) - set(split_df["path"])
    if missing:
        raise FileNotFoundError(f"{len(missing)} images in split.csv are missing from {DATA_DIR}")
    if new:
        print(f"NOTE: {len(new)} images not in split.csv — dataset may have changed since split was built.")
    print(f"Loaded {SPLIT_PATH}")

CLASSES = sorted(split_df["label"].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

counts = split_df.pivot_table(index="label", columns="split", aggfunc="size", fill_value=0)
counts = counts[["train", "val", "test"]]
counts.loc["TOTAL"] = counts.sum()
print(f"\n{NUM_CLASSES} classes, {len(split_df)} images total")
counts

## Data Loading + Augmentation

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ImageNet normalization stats

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class ObjectDataset(Dataset):
    def __init__(self, df, root, transform):
        self.paths = [Path(root) / p for p in df["path"]]
        self.labels = [CLASS_TO_IDX[l] for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = ImageOps.exif_transpose(im).convert("RGB")  # respects phone photo rotation
        return self.transform(img), self.labels[i]


def make_loader(split, transform, shuffle):
    ds = ObjectDataset(split_df[split_df["split"] == split], DATA_DIR, transform)
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, generator=g,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")


train_loader = make_loader("train", train_tf, shuffle=True)
val_loader = make_loader("val", eval_tf, shuffle=False)
test_loader = make_loader("test", eval_tf, shuffle=False)
print({k: len(l.dataset) for k, l in [("train", train_loader), ("val", val_loader), ("test", test_loader)]})

## Model Architectures

In [ ]:
class SimpleCNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        def block(c_in, c_out):
            return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out),
                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))

        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128),
                                      nn.AdaptiveAvgPool2d((4, 4)))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True),
                                        nn.Dropout(0.5), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(name, num_classes, freeze):
    if name == "simple_cnn":
        return SimpleCNN(num_classes), None
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        head = "fc."
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        head = "classifier."
    elif name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        head = "classifier."
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        head = "classifier."
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
        head = "classifier."
    else:
        raise ValueError(f"Unknown model: {name}")
    if freeze:
        for pname, p in m.named_parameters():
            p.requires_grad = pname.startswith(head)
    return m, head


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

## Training + Evaluation Functions

In [ ]:
def set_train_mode(model, freeze):
    model.train()
    if freeze:   # frozen backbone: keep BatchNorm running stats fixed too
        for mod in model.modules():
            if isinstance(mod, nn.modules.batchnorm._BatchNorm):
                mod.eval()


def run_epoch(model, loader, criterion, optimizer=None, freeze=False):
    training = optimizer is not None
    if training:
        set_train_mode(model, freeze)
    else:
        model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def predict(model, loader):
    model.eval()
    y_true, y_pred, conf = [], [], []
    for x, y in loader:
        probs = torch.softmax(model(x.to(DEVICE)), 1).cpu()
        y_true += y.tolist()
        y_pred += probs.argmax(1).tolist()
        conf += probs.max(1).values.tolist()
    return np.array(y_true), np.array(y_pred), np.array(conf)


@torch.no_grad()
def inference_ms_per_image(model, n=50):
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    for _ in range(10):
        model(x)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        model(x)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000


def plot_curves(history, tag):
    h = pd.DataFrame(history)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(h["epoch"], h["train_loss"], label="train")
    ax[0].plot(h["epoch"], h["val_loss"], label="val")
    ax[0].set(title=f"{tag} — loss", xlabel="epoch")
    ax[1].plot(h["epoch"], h["train_acc"], label="train")
    ax[1].plot(h["epoch"], h["val_acc"], label="val")
    ax[1].set(title=f"{tag} — accuracy", xlabel="epoch", ylim=(0, 1.02))
    for a in ax:
        a.legend()
        a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"curves_{tag}.png", dpi=150)
    plt.show()


def plot_confusion(cm, tag):
    size = max(5, 0.45 * NUM_CLASSES + 2)
    fig, ax = plt.subplots(figsize=(size, size))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES), CLASSES, rotation=90, fontsize=8)
    ax.set_yticks(range(NUM_CLASSES), CLASSES, fontsize=8)
    ax.set(xlabel="Predicted", ylabel="True", title=f"{tag} — test confusion matrix")
    if NUM_CLASSES <= 30:
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                if cm[i, j]:
                    ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                            color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"confusion_{tag}.png", dpi=150 if NUM_CLASSES <= 30 else 80)
    plt.show()


TAG_SUFFIX = {"scratch": "scratch", "frozen": "frozen", "finetune": "ft"}


def train_and_evaluate(name, freeze):
    if name == "simple_cnn":
        freeze, strategy = False, "scratch"
    else:
        strategy = "frozen" if freeze else "finetune"
    tag = f"{name}_{TAG_SUFFIX[strategy]}"
    print(f"\n{'=' * 70}\n{tag}  (lr={LR[strategy]})\n{'=' * 70}")

    seed_everything(SEED)
    model, _ = build_model(name, NUM_CLASSES, freeze)
    model.to(DEVICE)
    total_params, trainable_params = count_params(model)
    print(f"params: {total_params:,} total, {trainable_params:,} trainable")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR[strategy])
    weights_path = RESULTS_DIR / f"{tag}.pt"

    history, best_val, best_epoch, bad_epochs = [], float("inf"), 0, 0
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, freeze)
        va_loss, va_acc = run_epoch(model, val_loader, criterion)
        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc, val_loss=va_loss, val_acc=va_acc))
        improved = va_loss < best_val
        line = (f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                f"val loss {va_loss:.4f} acc {va_acc:.3f}" + ("  *" if improved else ""))
        print(line)
        with open(RESULTS_DIR / "train_log.txt", "a") as log:
            log.write(f"{time.strftime('%H:%M:%S')} {tag} {line}\n")
        if improved:
            best_val, best_epoch, bad_epochs = va_loss, epoch, 0
            torch.save(model.state_dict(), weights_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"early stopping (best epoch {best_epoch})")
                break
    train_time = time.time() - t0

    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    y_true, y_pred, conf = predict(model, test_loader)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))

    test_df = split_df[split_df["split"] == "test"].reset_index(drop=True)
    test_df = test_df.assign(true=[CLASSES[i] for i in y_true], pred=[CLASSES[i] for i in y_pred],
                             confidence=conf.round(4), correct=y_true == y_pred)
    test_df.to_csv(RESULTS_DIR / f"predictions_{tag}.csv", index=False)

    result = {
        "tag": tag, "model": name, "strategy": strategy, "freeze": freeze,
        "test_accuracy": accuracy_score(y_true, y_pred),
        "test_precision_macro": precision, "test_recall_macro": recall, "test_f1_macro": f1,
        "best_val_loss": best_val, "best_val_acc": history[best_epoch - 1]["val_acc"],
        "best_epoch": best_epoch, "epochs_run": len(history),
        "total_params": total_params, "trainable_params": trainable_params,
        "train_time_sec": round(train_time, 1),
        "inference_ms_per_image": round(inference_ms_per_image(model), 2),
        "device": str(DEVICE) + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""),
        "num_classes": NUM_CLASSES, "classes": CLASSES,
        "n_train": len(train_loader.dataset), "n_val": len(val_loader.dataset), "n_test": len(test_loader.dataset),
        "settings": {"img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "max_epochs": EPOCHS,
                     "patience": PATIENCE, "seed": SEED, "lr": LR[strategy], "optimizer": "Adam"},
        "history": history,
        "confusion_matrix": cm.tolist(),
        "per_class": classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=CLASSES,
                                           output_dict=True, zero_division=0),
    }
    with open(RESULTS_DIR / f"results_{tag}.json", "w") as f:
        json.dump(result, f, indent=2)

    print(f"\nTEST  acc {result['test_accuracy']:.4f} | F1 {f1:.4f} | precision {precision:.4f} | "
          f"recall {recall:.4f} | train {train_time / 60:.1f} min | {result['inference_ms_per_image']} ms/img")
    plot_curves(history, tag)
    plot_confusion(cm, tag)
    return result

## Baseline Models: Custom CNN & ConvNeXt-Tiny

In [ ]:
BASELINE_RUNS = [
    ("simple_cnn", False),
    ("convnext_tiny", False),
]

baseline_results = [train_and_evaluate(name, freeze) for name, freeze in BASELINE_RUNS]

baseline_summary = pd.DataFrame(baseline_results)[["tag", "test_accuracy", "test_f1_macro", "test_precision_macro",
                                                     "test_recall_macro", "trainable_params", "total_params",
                                                     "train_time_sec", "inference_ms_per_image", "best_epoch"]]
baseline_summary.round(4)

## ResNet50: Frozen vs Fine-Tuned

In [ ]:
result_resnet50_frozen = train_and_evaluate("resnet50", freeze=True)
result_resnet50_ft = train_and_evaluate("resnet50", freeze=False)

resnet_summary = pd.DataFrame([result_resnet50_frozen, result_resnet50_ft])[
    ["tag", "test_accuracy", "test_f1_macro", "test_precision_macro",
     "test_recall_macro", "trainable_params", "total_params",
     "train_time_sec", "inference_ms_per_image", "best_epoch"]
]
resnet_summary.round(4)

## EfficientNet-B0: Frozen vs Fine-Tuned

In [ ]:
result_efficientnet_frozen = train_and_evaluate("efficientnet_b0", freeze=True)
result_efficientnet_ft = train_and_evaluate("efficientnet_b0", freeze=False)

efficientnet_summary = pd.DataFrame([result_efficientnet_frozen, result_efficientnet_ft])[
    ["tag", "test_accuracy", "test_f1_macro", "test_precision_macro",
     "test_recall_macro", "trainable_params", "total_params",
     "train_time_sec", "inference_ms_per_image", "best_epoch"]
]
efficientnet_summary.round(4)

## MobileNetV2: Frozen vs Fine-Tuned

In [ ]:
result_mobilenet_frozen = train_and_evaluate("mobilenet_v2", freeze=True)
result_mobilenet_ft = train_and_evaluate("mobilenet_v2", freeze=False)

mobilenet_summary = pd.DataFrame([result_mobilenet_frozen, result_mobilenet_ft])[
    ["tag", "test_accuracy", "test_f1_macro", "test_precision_macro",
     "test_recall_macro", "trainable_params", "total_params",
     "train_time_sec", "inference_ms_per_image", "best_epoch"]
]
mobilenet_summary.round(4)

## DenseNet121: Fine-Tuned

In [ ]:
result_densenet_ft = train_and_evaluate("densenet121", freeze=False)

densenet_summary = pd.DataFrame([result_densenet_ft])[
    ["tag", "test_accuracy", "test_f1_macro", "test_precision_macro",
     "test_recall_macro", "trainable_params", "total_params",
     "train_time_sec", "inference_ms_per_image", "best_epoch"]
]
densenet_summary.round(4)

## Final Comparison — All Models

In [ ]:
import glob

all_results = []
for path in glob.glob(str(RESULTS_DIR / "results_*.json")):
    with open(path) as f:
        all_results.append(json.load(f))

comparison_df = pd.DataFrame(all_results)[["tag", "test_accuracy", "test_precision_macro",
                                             "test_recall_macro", "test_f1_macro",
                                             "total_params", "train_time_sec", "inference_ms_per_image"]]
comparison_df = comparison_df.sort_values("test_accuracy", ascending=False).reset_index(drop=True)
comparison_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison_df))
width = 0.35

ax.bar(x - width/2, comparison_df['test_accuracy'], width, label='Test Accuracy')
ax.bar(x + width/2, comparison_df['test_f1_macro'], width, label='F1 (macro)')

ax.set_xticks(x)
ax.set_xticklabels(comparison_df['tag'], rotation=30, ha='right')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Accuracy vs F1 (Macro)')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image as PILImage

curve_files = sorted(RESULTS_DIR.glob("curves_*.png"))
fig, axes = plt.subplots(len(curve_files), 1, figsize=(10, 4 * len(curve_files)))
if len(curve_files) == 1:
    axes = [axes]
for ax, f in zip(axes, curve_files):
    img = PILImage.open(f)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f.stem.replace('curves_', ''))
plt.tight_layout()
plt.show()

In [ ]:
best_row = comparison_df.iloc[0]
print(f"Best performing model: {best_row['tag']}")
print(f"  Test Accuracy: {best_row['test_accuracy']:.4f}")
print(f"  F1 (macro):    {best_row['test_f1_macro']:.4f}")
print(f"  Inference:     {best_row['inference_ms_per_image']:.2f} ms/image")
print(f"  Parameters:    {best_row['total_params']:,}")

print("\nFull ranked comparison:")
comparison_df